### Baseline interpretation

The majority-class baseline predicts every transaction as legitimate.
Because fraudulent transactions represent only a very small proportion
of the dataset, this produces extremely high overall accuracy despite
detecting no fraudulent transactions.

This demonstrates why accuracy is unsuitable as the primary evaluation
metric for FinGuard. Subsequent models will instead be evaluated using
fraud-class precision and recall, F1-score, PR-AUC and the associated
false-positive trade-off.

In [1]:
import pandas as pd
import numpy as np

DATA_PATH = "../data/raw/paysim.csv"

df = pd.read_csv(DATA_PATH)

print(df.shape)

(6362620, 11)


In [4]:
# origin balance error
df["orig_balance_error"] = (
    df["oldbalanceOrg"]
    - df["amount"]
    - df["newbalanceOrig"]
).abs()

df["amount_to_orig_balance"] = (
    df["amount"] /
    (df["oldbalanceOrg"] + 1)
)

In [3]:
# destination balance error
df["dest_balance_error"] = (
    df["oldbalanceDest"]
    + df["amount"]
    - df["newbalanceDest"]
).abs()

df["amount_to_dest_balance"] = (
    df["amount"] /
    (df["oldbalanceDest"] + 1)
)

In [5]:
df["hour"] = (df["step"] - 1) % 24
df["day"] = (df["step"] - 1) // 24

In [7]:
# what the modle is allowed to see
drop_columns = [
    "nameOrig",
    "nameDest",
    "isFlaggedFraud"
]

df_model = df.drop(columns=drop_columns)
df_model.columns.tolist()

['step',
 'type',
 'amount',
 'oldbalanceOrg',
 'newbalanceOrig',
 'oldbalanceDest',
 'newbalanceDest',
 'isFraud',
 'orig_balance_error',
 'dest_balance_error',
 'amount_to_dest_balance',
 'amount_to_orig_balance',
 'hour',
 'day']

In [8]:
df.groupby("isFraud")[
    [
        "orig_balance_error",
        "dest_balance_error",
        "amount_to_orig_balance",
        "amount_to_dest_balance"
    ]
].median()

,orig_balance_error,dest_balance_error,amount_to_orig_balance,amount_to_dest_balance
isFraud,,,,
0,69049.31,5123.10,6.511566,0.915134
1,0.00,9511.69,0.999998,116419.580000


In [9]:
df.groupby("isFraud")[
    [
        "orig_balance_error",
        "dest_balance_error"
    ]
].mean()

,orig_balance_error,dest_balance_error
isFraud,,
0,201338.558304,92756.964361
1,10692.325265,745138.585637


In [10]:
df[
    [
        "amount",
        "oldbalanceOrg",
        "newbalanceOrig",
        "oldbalanceDest",
        "newbalanceDest",
        "orig_balance_error",
        "dest_balance_error"
    ]
].describe()

,amount,oldbalanceOrg,newbalanceOrig,oldbalanceDest,newbalanceDest,orig_balance_error,dest_balance_error
count,6.362620e+06,6.362620e+06,6.362620e+06,6.362620e+06,6.362620e+06,6.362620e+06,6.362620e+06
mean,1.798619e+05,8.338831e+05,8.551137e+05,1.100702e+06,1.224996e+06,2.010925e+05,9.359907e+04
std,6.038582e+05,2.888243e+06,2.924049e+06,3.399180e+06,3.674129e+06,6.066505e+05,4.350570e+05
min,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
25%,1.338957e+04,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,2.954230e+03,0.000000e+00
50%,7.487194e+04,1.420800e+04,0.000000e+00,1.327057e+05,2.146614e+05,6.867726e+04,5.123620e+03
75%,2.087215e+05,1.073152e+05,1.442584e+05,9.430367e+05,1.111909e+06,2.496411e+05,4.342133e+04
max,9.244552e+07,5.958504e+07,4.958504e+07,3.560159e+08,3.561793e+08,9.244552e+07,7.588573e+07


In [11]:
df = pd.get_dummies(
    df,
    columns=["type"],
    prefix="type",
    dtype=int
)

In [12]:
df.columns.tolist()

['step',
 'amount',
 'nameOrig',
 'oldbalanceOrg',
 'newbalanceOrig',
 'nameDest',
 'oldbalanceDest',
 'newbalanceDest',
 'isFraud',
 'isFlaggedFraud',
 'orig_balance_error',
 'dest_balance_error',
 'amount_to_dest_balance',
 'amount_to_orig_balance',
 'hour',
 'day',
 'type_CASH_IN',
 'type_CASH_OUT',
 'type_DEBIT',
 'type_PAYMENT',
 'type_TRANSFER']

In [13]:
drop_columns = [
    "nameOrig",
    "nameDest",
    "isFlaggedFraud"
]

df_model = df.drop(columns=drop_columns)

In [14]:
split_step = int(df_model["step"].max() * 0.8)

split_step

594

In [15]:
train_df = df_model[df_model["step"] <= split_step].copy()
test_df = df_model[df_model["step"] > split_step].copy()

print("Training:", train_df.shape)
print("Testing:", test_df.shape)

print("\nTraining steps:",
      train_df["step"].min(),
      "to",
      train_df["step"].max())

print("Testing steps:",
      test_df["step"].min(),
      "to",
      test_df["step"].max())

Training: (6239040, 18)
Testing: (123580, 18)

Training steps: 1 to 594
Testing steps: 595 to 743


In [16]:
print("TRAIN")
print(train_df["isFraud"].value_counts())
print(train_df["isFraud"].value_counts(normalize=True) * 100)

print("\nTEST")
print(test_df["isFraud"].value_counts())
print(test_df["isFraud"].value_counts(normalize=True) * 100)

TRAIN
isFraud
0    6232481
1       6559
Name: count, dtype: int64
isFraud
0    99.894872
1     0.105128
Name: proportion, dtype: float64

TEST
isFraud
0    121926
1      1654
Name: count, dtype: int64
isFraud
0    98.661596
1     1.338404
Name: proportion, dtype: float64


In [17]:
print(
    "Training fraud cases:",
    train_df["isFraud"].sum()
)

print(
    "Testing fraud cases:",
    test_df["isFraud"].sum()
)

Training fraud cases: 6559
Testing fraud cases: 1654


In [19]:
# speerating x and y for training and testing
X_train = train_df.drop(columns=["isFraud"])
y_train = train_df["isFraud"]

X_test = test_df.drop(columns=["isFraud"])
y_test = test_df["isFraud"]

print(X_train.shape, y_train.shape)
print(X_test.shape, y_test.shape)

(6239040, 17) (6239040,)
(123580, 17) (123580,)


In [20]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    average_precision_score,
    roc_auc_score
)

dummy = DummyClassifier(strategy="most_frequent")

dummy.fit(X_train, y_train)

y_pred_dummy = dummy.predict(X_test)

In [21]:
print(classification_report(
    y_test,
    y_pred_dummy,
    digits=4
))

              precision    recall  f1-score   support

           0     0.9866    1.0000    0.9933    121926
           1     0.0000    0.0000    0.0000      1654

    accuracy                         0.9866    123580
   macro avg     0.4933    0.5000    0.4966    123580
weighted avg     0.9734    0.9866    0.9800    123580



/Users/maryamellathy/Desktop/FinGaurd/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/maryamellathy/Desktop/FinGaurd/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/maryamellathy/Desktop/FinGaurd/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(ave

In [22]:
confusion_matrix(
    y_test,
    y_pred_dummy
)

array([[121926,      0],
       [  1654,      0]])

In [23]:
from sklearn.metrics import accuracy_score

accuracy = accuracy_score(
    y_test,
    y_pred_dummy
)

print(f"Accuracy: {accuracy:.6f}")

Accuracy: 0.986616


In [24]:
#fraud recall 
cm = confusion_matrix(y_test, y_pred_dummy)

tn, fp, fn, tp = cm.ravel()

print("True negatives:", tn)
print("False positives:", fp)
print("False negatives:", fn)
print("True positives:", tp)

fraud_recall = tp / (tp + fn)

print(f"\nFraud recall: {fraud_recall:.4f}")

True negatives: 121926
False positives: 0
False negatives: 1654
True positives: 0

Fraud recall: 0.0000


### Model 1- logistic regression

In [25]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

In [27]:
logistic_model = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(
        max_iter=1000,
        random_state=42
    ))
])

logistic_model.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('scaler', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](17,)","['step','amount','oldbalanceOrg',...,'type_DEBIT','type_PAYMENT', 'type_TRANSFER']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,17
,"copy copy: bool, default=TrueIf False, try to avoid a copy and do inplace scaling instead.This is not guaranteed to always work inplace; e.g. if the data isnot a NumPy array or scipy.sparse CSR matrix, a copy may still bereturned.",True
,"with_mean with_mean: bool, default=TrueIf True, center the data before scaling.This does not work (and will raise an exception) when attempted onsparse matrices, because centering them entails building a densematrix which in common use cases is likely to be too large to fit inmemory.",True
,"with_std with_std: bool, default=TrueIf True, scale the data to unit variance (or equivalently,unit standard deviation).",True


In [28]:
y_pred_logistic = logistic_model.predict(X_test)
y_prob_logistic = logistic_model.predict_proba(X_test)[:, 1]

In [29]:
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    average_precision_score,
    roc_auc_score
)

print(classification_report(
    y_test,
    y_pred_logistic,
    digits=4
))

              precision    recall  f1-score   support

           0     0.9938    0.9998    0.9968    121926
           1     0.9738    0.5393    0.6942      1654

    accuracy                         0.9936    123580
   macro avg     0.9838    0.7696    0.8455    123580
weighted avg     0.9935    0.9936    0.9927    123580



In [30]:
cm_logistic = confusion_matrix(
    y_test,
    y_pred_logistic
)

cm_logistic

array([[121902,     24],
       [   762,    892]])

In [31]:
pr_auc_logistic = average_precision_score(
    y_test,
    y_prob_logistic
)

roc_auc_logistic = roc_auc_score(
    y_test,
    y_prob_logistic
)

print(f"PR-AUC:  {pr_auc_logistic:.4f}")
print(f"ROC-AUC: {roc_auc_logistic:.4f}")

PR-AUC:  0.8148
ROC-AUC: 0.9846


In [33]:
weighted_logistic_model = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(
        class_weight="balanced",
        max_iter=1000,
        random_state=42
    ))
])

weighted_logistic_model.fit(
    X_train,
    y_train
)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('scaler', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](17,)","['step','amount','oldbalanceOrg',...,'type_DEBIT','type_PAYMENT', 'type_TRANSFER']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,17
,"copy copy: bool, default=TrueIf False, try to avoid a copy and do inplace scaling instead.This is not guaranteed to always work inplace; e.g. if the data isnot a NumPy array or scipy.sparse CSR matrix, a copy may still bereturned.",True
,"with_mean with_mean: bool, default=TrueIf True, center the data before scaling.This does not work (and will raise an exception) when attempted onsparse matrices, because centering them entails building a densematrix which in common use cases is likely to be too large to fit inmemory.",True
,"with_std with_std: bool, default=TrueIf True, scale the data to unit variance (or equivalently,unit standard deviation).",True


In [34]:
y_pred_weighted = weighted_logistic_model.predict(
    X_test
)

y_prob_weighted = weighted_logistic_model.predict_proba(
    X_test
)[:, 1]

In [35]:
print(classification_report(
    y_test,
    y_pred_weighted,
    digits=4
))

              precision    recall  f1-score   support

           0     0.9999    0.9153    0.9558    121926
           1     0.1373    0.9933    0.2413      1654

    accuracy                         0.9164    123580
   macro avg     0.5686    0.9543    0.5985    123580
weighted avg     0.9884    0.9164    0.9462    123580



In [36]:
cm_weighted = confusion_matrix(
    y_test,
    y_pred_weighted
)

cm_weighted

array([[111603,  10323],
       [    11,   1643]])

In [37]:
pr_auc_weighted = average_precision_score(
    y_test,
    y_prob_weighted
)

roc_auc_weighted = roc_auc_score(
    y_test,
    y_prob_weighted
)

print(f"PR-AUC:  {pr_auc_weighted:.4f}")
print(f"ROC-AUC: {roc_auc_weighted:.4f}")

PR-AUC:  0.8400
ROC-AUC: 0.9940


## Logistic Regression Baseline

Logistic Regression was selected as the first supervised baseline due
to its simplicity and interpretability.

Because fraud represents approximately 0.13% of the complete PaySim
dataset, both standard and class-weighted variants are evaluated.

The class-weighted model assigns greater importance to observations
from the minority fraud class during optimisation. The objective is
not simply to maximise overall accuracy, but to investigate whether
minority-class detection can be improved without producing an
unacceptable number of false positives.

Performance is evaluated using fraud precision, recall, F1-score,
PR-AUC and ROC-AUC.

In [40]:
# results table:
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    accuracy_score
)

results = pd.DataFrame({
    "Model": [
        "Dummy Baseline",
        "Logistic Regression",
        "Weighted Logistic Regression"
    ],

    "Accuracy": [
        accuracy_score(y_test, y_pred_dummy),
        accuracy_score(y_test, y_pred_logistic),
        accuracy_score(y_test, y_pred_weighted)
    ],

    "Fraud Precision": [
        precision_score(
            y_test,
            y_pred_dummy,
            zero_division=0
        ),
        precision_score(
            y_test,
            y_pred_logistic,
            zero_division=0
        ),
        precision_score(
            y_test,
            y_pred_weighted,
            zero_division=0
        )
    ],

    "Fraud Recall": [
        recall_score(y_test, y_pred_dummy),
        recall_score(y_test, y_pred_logistic),
        recall_score(y_test, y_pred_weighted)
    ],

    "Fraud F1": [
        f1_score(y_test, y_pred_dummy),
        f1_score(y_test, y_pred_logistic),
        f1_score(y_test, y_pred_weighted)
    ]
})

results

,Model,Accuracy,Fraud Precision,Fraud Recall,Fraud F1
0,Dummy Baseline,0.986616,0.000000,0.000000,0.000000
1,Logistic Regression,0.993640,0.973799,0.539299,0.694163
2,Weighted Logistic Regression,0.916378,0.137306,0.993349,0.241263
